# Notebook 2 - Agent Definition
Owner: Hyunju Yu - AI Engineer

---

### What this notebook does

This notebook runs the PaySprint investment agent end-to-end.
The agent follows the ReAct pattern: it reasons about what to do, calls a tool,
looks at the result, then decides what to do next - repeating until it writes the final report.

How to use this notebook:
1. Add your OpenAI API key in the first cell (required)
2. Run every cell from top to bottom
3. Look for the sections marked YOUR TURN - those are yours to customize

---

### Before you start - API Key Setup

You need an OpenAI API key. Get one at https://platform.openai.com/api-keys

Never paste your key directly into a shared notebook.
Instead, create a file called .env in the project folder with this line:

    OPENAI_API_KEY=sk-your-key-here

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'])

from dotenv import load_dotenv
load_dotenv()

import os
if os.getenv('OPENAI_API_KEY'):
    print('API key loaded from .env file')
else:
    from getpass import getpass
    os.environ['OPENAI_API_KEY'] = getpass('Paste your OpenAI API key here: ')
    print('API key set for this session')

In [ ]:
import sys
sys.path.insert(0, '.')

from paysprint_agent import (
    run_agent,
    test_rejection,
    print_trace_summary,
    estimate_cost,
    save_trace,
    SCREENER_STOCKS,
    SCORING_WEIGHTS,
    MODEL_REASONING,
    MODEL_SUMMARY,
)
import paysprint_agent as core
import os
print(f'Agent loaded  |  Reasoning model: {MODEL_REASONING}  |  Summary model: {MODEL_SUMMARY}')

---
## Step 1 - Choose the LLMs

The agent uses two different LLMs (required by the project rubric):
- MODEL_REASONING: the smarter, more expensive model that runs the orchestrator and writes the final report
- MODEL_SUMMARY: the faster, cheaper model used for evaluation

Both are used in the same trace, which satisfies the rubric requirement.

In [ ]:
# YOUR TURN: You can change the model names here if you want to experiment
#
# Available OpenAI models:
#   'gpt-4o'       - best quality, more expensive  (~$2.50 per 1M input tokens)
#   'gpt-4o-mini'  - good quality, much cheaper    (~$0.15 per 1M input tokens)
#   'gpt-4.1'      - newer, strong reasoning
#   'gpt-4.1-mini' - newer, cheaper option

MODEL_1 = 'gpt-4o'       # main agent
MODEL_2 = 'gpt-4o-mini'  # comparison model

print(f'Model 1 (reasoning): {MODEL_1}')
print(f'Model 2 (comparison): {MODEL_2}')
print()
print('Pricing reference:')
for model, prices in core.MODEL_PRICES.items():
    print(f'  {model}: ${prices["input"]}/1M input, ${prices["output"]}/1M output')

---
## Step 2 - Run the Agent (Demo)

This is the full agent run. It will:
1. Screen stocks for the user strategy
2. Fetch technical indicators, news sentiment, and fundamentals for each stock
3. Build a purchase plan
4. Write a final investment report

This takes 1-3 minutes depending on API response times.
You will see each tool call printed as it runs.

In [ ]:
DEMO_PROFILE = {
    'name':              'Demo Investor',
    'budget':            5000.0,
    'aggressiveness':    'moderate',
    'horizon_months':    12,
    'current_holdings':  {},
    'preferred_sectors': ['Technology'],
}

print('Running PaySprint agent...\n')
print(f'Profile: ${DEMO_PROFILE["budget"]:,.0f} budget | {DEMO_PROFILE["aggressiveness"]} strategy | {DEMO_PROFILE["horizon_months"]}mo horizon')
print('-' * 60)

result = run_agent(DEMO_PROFILE, model=MODEL_1, verbose=True)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(result['report'])

In [ ]:
print_trace_summary(result)

cost = estimate_cost(result)
print('\nCost estimate:')
print(f'  Model:          {cost["model"]}')
print(f'  Input tokens:   {cost["input_tokens"]:,}')
print(f'  Output tokens:  {cost["output_tokens"]:,}')
print(f'  Estimated cost: ${cost["cost_usd"]:.4f} USD')

In [ ]:
result['trace_id'] = 1
result['label']    = 'Demo run - moderate $5k'
result['profile']  = DEMO_PROFILE
save_trace(result, trace_id=1)
print('Saved as Trace 1')

---
## Step 3 - Test Graceful Rejection

The agent should politely refuse irrelevant questions without calling any tools.
These two tests are required by the project rubric.

In [ ]:
rejection_tests = [
    'What is the capital of France?',
    'Can you write me a recipe for pasta?',
]

print('Testing graceful rejection...\n')
rejection_results = []
for msg in rejection_tests:
    r = test_rejection(msg, model=MODEL_1)
    rejection_results.append(r)
    status = 'PASS (no tools called)' if not r['tool_calls_made'] else 'FAIL (tools were called)'
    print(f'Input:    "{msg}"')
    print(f'Response: {r["response"][:200]}')
    print(f'Result:   {status}')
    print()

---
## YOUR TURN - Hyunju's Customizations

Everything above already works. The tasks below are yours to explore and improve.

---
### Task A - Customize the agent instructions

The system prompt tells the agent how to behave.
You can change its tone, add requirements, or adjust the report format.

Recommendation: Change the writing style and add one extra requirement to the report.

In [ ]:
# Show the current system prompt so you can read it before editing
from paysprint_agent import build_system_prompt
print('Current system prompt (first 600 chars):')
print(build_system_prompt(DEMO_PROFILE)[:600])
print('...')

In [ ]:
import json as _json

# YOUR TURN: Edit the text inside the triple quotes below
# Keep the {profile.get(...)} references - they fill in the user's actual data
# Ideas:
#   - Make the tone more or less formal
#   - Add: 'Always flag stocks with negative revenue_growth with a WARNING note'
#   - Add: 'Rate each stock as LOW RISK, MEDIUM RISK, or HIGH RISK at the top of its summary'
#   - Change how the final report is formatted (bullet points, numbered list, table)

def my_custom_system_prompt(profile: dict) -> str:
    return f"""You are PaySprint, a friendly and professional AI investment research assistant.
Your goal is to help everyday investors make confident, well-informed decisions.
Always write clearly - assume the user is not a finance expert.

User Profile:
  - Name:     {profile.get('name', 'Investor')}
  - Budget:   ${profile.get('budget', 0):,.2f}
  - Strategy: {profile.get('aggressiveness', 'moderate')}
  - Horizon:  {profile.get('horizon_months', 12)} months

Follow this workflow:
1. Call screen_stocks for the user strategy.
2. For each stock: call get_technical_indicators, get_news_sentiment, get_fundamentals.
3. Pick the best 3-5 stocks and explain why in plain English.
4. Call create_purchase_plan and show an easy-to-read table.
5. End with a RISK DISCLOSURE (required).

Extra requirements added by Hyunju:
  - If a stock has negative revenue_growth, add a WARNING next to its name.
  - Rate each stock CONSERVATIVE, MODERATE, or AGGRESSIVE at the top of its summary.
  - Use bullet points for stock summaries - keep each one under 3 bullets.

If the user asks anything unrelated to investing, politely decline.
"""

core.build_system_prompt = my_custom_system_prompt
print('Custom prompt applied. Run the next cell to test it.')

In [ ]:
print('Running agent with custom prompt...\n')
custom_result = run_agent(DEMO_PROFILE, model=MODEL_1, verbose=True)
print('\n' + '=' * 60)
print('CUSTOM PROMPT REPORT:')
print('=' * 60)
print(custom_result['report'])

---
### Task B - Adjust the scoring weights

The agent scores stocks using three signals: sentiment, momentum, and news mentions.
You can change how much each signal matters.

Recommendation: Try making momentum more important for aggressive strategies.

In [ ]:
print('Current scoring weights:')
for k, v in core.SCORING_WEIGHTS.items():
    print(f'  {k}: {v}')
print(f'  Total: {sum(core.SCORING_WEIGHTS.values()):.2f}  (must equal 1.0)')

In [ ]:
# YOUR TURN: Change the three values below
# They must add up to 1.0
# Think about which signals matter most for each strategy:
#   - sentiment  = how positive is the news?
#   - momentum   = is the price trending up?
#   - mentions   = how much news coverage does the stock get?

core.SCORING_WEIGHTS = {
    'sentiment': 0.35,
    'momentum':  0.45,
    'mentions':  0.20,
}

total = sum(core.SCORING_WEIGHTS.values())
assert abs(total - 1.0) < 0.001, f'Weights must sum to 1.0, got {total}'
print(f'Weights updated. Total = {total:.2f}')
for k, v in core.SCORING_WEIGHTS.items():
    print(f'  {k}: {v}')

---
### Task C - Write your agent behavior observations

Hyunju: Replace the placeholder text below with your own observations.

After running the agent, write 3-5 sentences answering:
- Did the agent call all the tools it should have?
- Was the final report easy to understand?
- Did the recommendations match the strategy (moderate / conservative / aggressive)?
- Did your custom prompt change the output in a useful way?
- What would you improve about the agent if you had more time?

This commentary is required by the rubric.

**Hyunju's Agent Performance Commentary**

Write your observations here after running the agent.

Example: The agent correctly called all 5 tools and generated a clear report.
The moderate strategy selected AAPL, MSFT, and GOOGL, which are appropriate choices.
After applying the custom prompt, the report used bullet points which made it easier to read.
The rejection test worked correctly - the agent refused both off-topic questions without calling any tools.